# Luka's Log

In [1]:
import json
offical_run = json.load(open("LSC23.json"))
# 'id', 'name', 'description', 'started', 'ended', 'tasks', 'hasStarted', 'running', 'hasEnded'

In [2]:
descriptions = offical_run['description']
# 'id', 'name', 'description', 'taskTypes', 'taskGroups', 'tasks', 'teams', 'teamGroups', 'participantCanView'
tasks = descriptions['tasks']
# list of ['id', 'name', 'taskGroup', 'taskType', 'duration', 'mediaCollectionId', 'target', 'hints']
targets = {}
queries = {}
for i in range(len(tasks)):
    if "KIS" in tasks[i]['name']:
        targets[tasks[i]['name']] = [item['location'] for item in tasks[i]['target']['items']]
        queries[tasks[i]['name']] = [item['text'] for item in tasks[i]['hints']]
    else:
        targets[tasks[i]['name']] = []
        queries[tasks[i]['name']] = [item['text'] for item in tasks[i]['hints']]
print(len(targets), len(queries))

63 63


In [3]:
teams = offical_run['description']['teams']
# list of ['uid', 'name', 'color', 'logoId', 'users']
team_ids = {item["uid"]["string"]: item["name"] for item in teams}

In [4]:
TEAMS = list(team_ids.values())
TASKS = ["KIS", "QA", "AD"]

def task_type(name):
    if "QA" in name:
        return "QA"
    elif "KIS" in name:
        return "KIS"
    return "AD"

def make_task_dicts(data_type):
    # deep copy obj to 3 keys: AD, QA, KIS
    return {task: defaultdict(data_type) for task in TASKS}

In [5]:
tasks = offical_run['tasks']
print("Number of tasks: ", len(tasks))
# list of ['started', 'ended', 'submissions', 'description', 'filter', 'scorer', 'validator', 'duration', 'uid',
#          'taskDescriptionId', 'position', 'hasStarted', 'running', 'hasEnded', 'teamGroupAggregators']
# ['teamId', 'memberId', 'timestamp', 'item', 'uid', 'status']
submissions = {}
start_times = {}
mysceal = []
names = []
real_queries = {}
real_targets = {}
for i in range(len(tasks)):
    name = tasks[i]["description"]["name"]
    if name in queries:
        names.append(name)
        real_queries[name] = queries[name]
        real_targets[name] = targets[name]
        start_times[name] = tasks[i]["started"]
        if task_type(name) != "QA":
            submissions[name] = [(team_ids[item["teamId"]["string"]],
                            item["status"],
                            (item["timestamp"] - start_times[name])/1000,
                            item["item"]["name"]) for item in tasks[i]['submissions']]
        else:
            submissions[name] = [(team_ids[item["teamId"]["string"]],
                            item["status"],
                            (item["timestamp"] - start_times[name])/1000,
                            item["text"]) for item in tasks[i]['submissions']]
print(names)


Number of tasks:  30
['LSC23-KIS01', 'LSC23-AD01', 'LSC23-QA01', 'LSC23-KIS02', 'LSC23-AD02', 'LSC23-QA02', 'LSC23-KIS03', 'LSC23-AD03', 'LSC23-QA03', 'LSC23-KIS04', 'LSC23-AD04', 'LSC23-QA04', 'LSC23-KIS05', 'LSC23-AD05', 'LSC23-QA05', 'LSC23-KIS06', 'LSC23-AD06', 'LSC23-QA06', 'LSC23-KIS07N', 'LSC23-AD07N', 'LSC23-QA08N', 'LSC23-KIS08N', 'LSC23-AD08N', 'LSC23-QA09N', 'LSC23-KIS09N', 'LSC23-AD09N', 'LSC23-QA10N', 'LSC23-KIS10N', 'LSC23-AD10N', 'LSC23-QA07N']


## Correct/Incorrect

In [6]:
from collections import defaultdict
correct_counts = make_task_dicts(int)
incorrect_counts = make_task_dicts(int)
total_counts = make_task_dicts(int)
time_till_correct = defaultdict(lambda: defaultdict(lambda: None))
time_till_correct_full = defaultdict(lambda: defaultdict(lambda: 300))
top_3 = make_task_dicts(int)
limits = {"QA": 180, "KIS": 300, "AD": 180}

for team in TEAMS:
    for name in names:
        time_till_correct_full[team][name] = limits[task_type(name)]

for name in submissions:
    correct_time = 0
    task = task_type(name) 
    for team, status, time, image in submissions[name]:
        if status == "CORRECT":
            correct_counts[task][team] += 1
            time = min(time, limits[task])
            if time_till_correct[team][name]:
                time_till_correct[team][name] = min(time_till_correct[team][name], time)
            else:
                time_till_correct[team][name] = time
            
            time_till_correct_full[team][name] = min(time_till_correct_full[team][name], time)

            correct_time += 1
            if correct_time <= 3:
                top_3[task][team] += 1
        else:
            incorrect_counts[task][team] += 1

        total_counts[task][team] += 1

In [ ]:
# Significance test for speed
from scipy.stats import ttest_ind

for cur_task in TASKS:
    print(cur_task)
    myEachtra_scores = [time for task, time in time_till_correct_full["MyEachtra"].items() if task_type(task) == cur_task]
    print(myEachtra_scores)
    other_scores = []
    for team in TEAMS:
        if team != "MyEachtra":
            other_scores += [time for task, time in time_till_correct_full[team].items() if task_type(task) == cur_task]
    mySceal_score = [time for task, time in time_till_correct_full["2022 Baseline"].items() if task_type(task) == cur_task]

    print(ttest_ind(myEachtra_scores, other_scores))
    print(ttest_ind(myEachtra_scores, mySceal_score))
    print(ttest_ind(mySceal_score, other_scores))


### Scoring

In [7]:
import math
import numpy as np
# Scores for each queries
max_point = 100
max_point_end = 50
penalty = 10
scores = {"QA": [],
          "AD": [],
          "KIS": []}
recall_ad = defaultdict(list)
precision_ad = defaultdict(list)

for name, submission in submissions.items():
    task = task_type(name)
    if task == "AD":
        correct = defaultdict(int)
        incorrect = defaultdict(int)
        score = defaultdict(float)
        num_correct = set()
        penalty = 0.2
        
        max_time = 180
        for team, status, time, image in submission:
            if status == "WRONG":
                incorrect[team] += 1
            elif status == "CORRECT":
                correct[team] += 1
                num_correct.add(image)
                real_targets[name].append(image)
                
        for team in TEAMS:
            if correct[team] == 0:
                score[team] = 0
                recall_ad[team].append(0)
                precision_ad[team].append(0)
            else:
                score[team] = correct[team] * max_point / (correct[team] + incorrect[team]/2) * correct[team] / len(num_correct)
                recall_ad[team].append(correct[team] / len(num_correct))
                precision_ad[team].append(correct[team] / (correct[team] + incorrect[team]))
    else:
        score = defaultdict(float)
        for team, status, time, image in submission:
            if status == "WRONG":
                score[team] -= penalty
            elif status == "CORRECT":
                score[team] += max_point_end + (max_point - max_point_end) * (1 - time/limits[task])
        for team in TEAMS:
            score[team] = max(0, score[team])
    scores[task].append(score)

precision_ad = {team: np.mean(precision_ad[team]) for team in TEAMS}
recall_ad = {team: np.mean(recall_ad[team]) for team in TEAMS}


In [9]:
# Export to json real tasks and their targets
data = []
for i in range(len(names)):
    data.append(
        {
            "name": names[i],
            "queries": real_queries[names[i]],
            "targets": real_targets[names[i]],
            "submissions": submissions[names[i]],
        }
    )
json.dump(data, open("LSC23_tasks.json", "w"), indent=2)

In [ ]:
# Significance test for scores of MyEachtra and others
from scipy.stats import ttest_ind

for task in TASKS:
    print(task)
    myEachtra_score = [score["MyEachtra"] for score in scores[task]]
    other_scores = []
    for submission in scores[task]:
        other_scores.extend([score for team, score in submission.items() if team != "MyEachtra"])
    print(ttest_ind(myEachtra_score, other_scores))
    # with mysceal
    mySceal_score = [score["MySceal"] for score in scores[task]]
    print(ttest_ind(myEachtra_score, mySceal_score))

In [ ]:
from collections import defaultdict, OrderedDict
TASKS = ["KIS", "QA", "AD"]
normalize_scores = {"QA": {},
                    "AD": {},
                    "KIS": {},
                    "SUM": defaultdict(float)}
for task in TASKS:
    cum_scores = defaultdict(float)
    for score in scores[task]: # iterate over all tasks in this task_type
        for team in score:
            cum_scores[team] += score[team]
                
    max_score = max(cum_scores.values())
    for team in cum_scores:
        normalize_scores[task][team] = round(cum_scores[team]/max_score * 100)
        normalize_scores["SUM"][team] += normalize_scores[task][team]
        
# sort TEAMS by normalize_scores
TEAMS = sorted(TEAMS, key=lambda team: normalize_scores["SUM"][team], reverse=True)
for task in TASKS:
    # sort dict by TEAMS
    normalize_scores[task] = OrderedDict(sorted(normalize_scores[task].items(), key=lambda x: TEAMS.index(x[0])))
print([(team, normalize_scores["SUM"][team]) for team in TEAMS])

In [ ]:
normalize_scores["AD"]

# GRAPHS

In [ ]:
import pandas as pd
import seaborn as sns

# Each row is a task corresponding to a team
# Columns: team, task, task_type, score, incorrect, correct, time_till_correct
detailed_df = pd.DataFrame(
    columns=[
        "team",
        "task",
        "task_type",
        "score",
        "incorrect",
        "correct",
        "time_till_correct",
    ]
)
for name, submission in submissions.items():
    task = task_type(name)
    correct = defaultdict(int)
    incorrect = defaultdict(int)
    if task == "AD":
        score = defaultdict(float)
        num_correct = set()
        penalty = 0.2
        max_time = 180
        for team, status, time, image in submission:
            if status == "WRONG":
                incorrect[team] += 1
            elif status == "CORRECT":
                correct[team] += 1
                num_correct.add(image)

        for team in TEAMS:
            if correct[team] == 0:
                score[team] = 0
            else:
                score[team] = (
                    correct[team]
                    * max_point
                    / (correct[team] + incorrect[team] / 2)
                    * correct[team]
                    / len(num_correct)
                )
    else:
        score = defaultdict(float)
        for team, status, time, image in submission:
            if status == "WRONG":
                score[team] -= penalty
                incorrect[team] += 1
            elif status == "CORRECT":
                score[team] += max_point_end + (max_point - max_point_end) * (
                    1 - time / limits[task]
                )
                correct[team] += 1
        for team in TEAMS:
            score[team] = max(0, score[team])
    for team in TEAMS:
        detailed_df = detailed_df.append(
            {
                "team": team,
                "task": name,
                "task_type": task,
                "score": score[team],
                "incorrect": incorrect[team],
                "correct": correct[team],
                "time_till_correct": time_till_correct[team][name],
            },
            ignore_index=True,
        )
detailed_df.to_csv("detailed_scores.csv")

In [ ]:
import pandas as pd
import seaborn as sns


df = None
for task in TASKS:
    data = {
        "team": TEAMS,
        "task": [f"{task}" for team in TEAMS],
        f"correct": [correct_counts[task][team] for team in TEAMS],
        f"incorrect": [incorrect_counts[task][team] for team in TEAMS],
        
        # f"cum_score_{task}": [cum_scores[task][team] for team in TEAMS],
        f"norm_score": [normalize_scores[task][team] for team in TEAMS],
    }
    if task != "AD":
        data.update({
            f"top_3": [top_3[task][team] for team in TEAMS],
            f"precision": [correct_counts[task][team]/total_counts[task][team] for team in TEAMS],
            f"recall": [correct_counts[task][team]/len(scores[task]) for team in TEAMS]})
    else:
        data["recall"] = [recall_ad[team] for team in TEAMS]
        data["precision"] = [precision_ad[team] for team in TEAMS]
    for i in time_till_correct["MyEachtra"]:
        data[f"time_correct_{i}"] = [time_till_correct[team][i] for team in TEAMS]
    for i in time_till_correct_full["MyEachtra"]:
        data[f"time_correct_full_{i}"] = [time_till_correct_full[team][i] for team in TEAMS]
    if df is None:
        df = pd.DataFrame(data=data)
    else:
        df = pd.concat([df, pd.DataFrame(data=data)], axis=0)   
        
# Change team "2022 Baseline" to "E-Myscéal"   
df["team"] = df["team"].apply(lambda x: "E-Myscéal" if x == "2022 Baseline" else x)
TEAMS = [t if t != "2022 Baseline" else "E-Myscéal" for t in TEAMS]
df

In [ ]:
df.to_csv('lsc.table', index=False)

In [ ]:
task_names = ["KIS", "AD"]

import matplotlib.pyplot as plt

sns.set_style("white")
plt.rcParams.update({'font.size': 16})
all_score = df.groupby(['team', 'task']).sum().reset_index().pivot(index='team', columns='task', values='norm_score')

# Sort task by ["KIS", "AD", "AD"]
all_score = all_score.reindex(columns=task_names)

# sort by the other in TEAMS
all_score = all_score.loc[TEAMS[::-1]]
fig = all_score.plot.barh(figsize=(15, 5), rot=0, stacked=True, 
                    title="Scores in KIS and Adhoc tasks in LSC'23", width=0.8,
                    color=["#98D2EB", "#B2B1CF", "#b0c5aa"], legend=True)

fig.bar_label(fig.containers[0], labels=all_score["KIS"], label_type='center')
# fig.bar_label(fig.containers[1], labels=all_score["QA"], label_type='center')
fig.bar_label(fig.containers[1], labels=all_score["AD"], label_type='center')

# Highlighting MyEachtra and E-Mysceal by fontweight
fig.axes.get_yticklabels()[-1].set_fontweight("bold")
fig.axes.get_yticklabels()[-6].set_fontweight("bold")

plt.savefig("overall_23.png", format="png", bbox_inches='tight')

In [ ]:
# Significance test for scores, MyEachtra vs. the rest
from scipy.stats import ttest_ind
for task in task_names:
    # get score where team = myeachtra and task = task
    myeachtra_score = df[(df["team"] == "MyEachtra") & (df["task"] == task)]["norm_score"]
    print(task)
    print(myeachtra_score)
    for team in TEAMS:
        if team == "MyEachtra":
            continue
        team_score = df[(df["team"] == team) & (df["task"] == task)]["norm_score"]
        print(f"Significance test for {team} vs. MyEachtra in {task} task: ", ttest_ind(myeachtra_score, team_score, equal_var=False))

In [ ]:
task_names = ["QA"]

import matplotlib.pyplot as plt

sns.set_style("white")
plt.rcParams.update({'font.size': 16})
all_score = df.groupby(['team', 'task']).sum().reset_index().pivot(index='team', columns='task', values='norm_score')

# Sort task by ["KIS", "AD", "AD"]
all_score = all_score.reindex(columns=task_names)

# sort by the other in TEAMS
all_score = all_score.loc[TEAMS[::-1]]
fig = all_score.plot.barh(figsize=(15, 5), rot=0, stacked=True, 
                    title="Scores in QA tasks in LSC'23", width=0.8,
                    color=["#b0c5aa"], legend=True)

fig.bar_label(fig.containers[0], labels=all_score["QA"], label_type='center')

# Highlighting MyEachtra and E-Mysceal by fontweight
fig.axes.get_yticklabels()[-1].set_fontweight("bold")
fig.axes.get_yticklabels()[-6].set_fontweight("bold")

plt.savefig("overall_qa_23.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16, 'figure.figsize': (20, 8)})
task_names = ["KIS", "AD"]
# Create subplots
fig, axes = plt.subplots(1, len(task_names), figsize=(16, 8))
# Plot data for each TASK
for i, TASK in enumerate(task_names):
    filtered = df[df["task"] == TASK]
    ax = filtered[["team", "correct", "incorrect"]].plot(x='team', kind='bar', stacked=True, width=0.8, color=["#8da0cb", "#fc8d62"], ax=axes[i])
    ax.bar_label(ax.containers[0], labels=filtered["correct"], label_type='center')
    ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["incorrect"]], label_type='center')
    
    ax.set_xlabel("Teams")
    # ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
    ax.set_xticklabels(filtered["team"])
    if i == 0:
        ax.text(-0.12, 0.5, "Number of Submissions", va='center', rotation='vertical', transform=ax.transAxes)

    ax.get_xticklabels()[0].set_fontweight("bold")
    ax.get_xticklabels()[5].set_fontweight("bold")
    # make y-axis ends at 12
    # ax.set_ylim(0, 12)
    
    # Turn off legend
    ax.legend().set_visible(False)

    # Title of the subplot
    ax.set_title(f"{TASK}", fontweight='bold')

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles[::-1], ["Incorrect", "Correct"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
fig.suptitle("Number of incorrect and correct queries per team in LSC'23", fontweight='bold')
plt.tight_layout()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("incorrect_correct_23.png", format="pdf", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# set figure size
plt.rcParams["figure.figsize"] = (12, 7)

# Plot data for each TASK
filtered = df[df["task"] == "AD"]
fig = filtered[["team", "precision", "recall"]].plot(x='team', kind='bar', width=0.8, color=["#8da0cb", "#fc8d62"])
fig.bar_label(fig.containers[0], labels=filtered["precision"].round(2), label_type='edge', fontsize=12)
fig.bar_label(fig.containers[1], labels=["" if x == 0 else x for x in filtered["recall"].round(2)], label_type='edge',fontsize=11)

plt.xlabel("Team", fontweight='bold')
fig.axes.get_xticklabels()[0].set_fontweight("bold")
fig.axes.get_xticklabels()[5].set_fontweight("bold")
# ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')f
# make y-axis ends at 12
# ax.set_ylim(0, 12)

# Turn off legend
fig.legend().set_visible(False)

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
plt.title("Precision and Recall per team in LSC'23 for Ad-hoc tasks.", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("precision-recall_23.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# set figure size
plt.rcParams["figure.figsize"] = (12, 7)

# Plot data for each TASK
filtered = df[df["task"] == "QA"]
fig = filtered[["team", "precision", "recall"]].plot(x='team', kind='bar', width=0.8, color=["#8da0cb", "#fc8d62"])
fig.bar_label(fig.containers[0], labels=filtered["precision"].round(2), label_type='edge', fontsize=10)
fig.bar_label(fig.containers[1], labels=["" if x == 0 else x for x in filtered["recall"].round(2)], label_type='edge',fontsize=10)

plt.xlabel("Team", fontweight='bold')
fig.axes.get_xticklabels()[0].set_fontweight("bold")
fig.axes.get_xticklabels()[5].set_fontweight("bold")
# ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')f
# make y-axis ends at 12
# ax.set_ylim(0, 12)

# Turn off legend
fig.legend().set_visible(False)

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
plt.title("Precision and Recall per team in LSC'23 for QA tasks.", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("precision-recall_23.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16, 'figure.figsize': (20, 8)})
filtered = df[df["task"] == "QA"]
# Create subplots
# fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# ================== Subplot 1 ================== #
i = 0 # Subplot 1, incorrect correct
plot1 = plt.subplot2grid((1, 5), (0, 0), colspan=2)

ax = filtered[["team", "correct", "incorrect"]].plot(x='team', kind='bar', stacked=True, width=0.8, color=["#8da0cb", "#fc8d62"], ax=plot1)
ax.bar_label(ax.containers[0], labels=filtered["correct"], label_type='center')
ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["incorrect"]], label_type='center')

ax.set_xlabel("Teams")
# ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
ax.set_xticklabels(filtered["team"])
ax.set_ylabel("Number of Submissions", fontweight='bold')

ax.get_xticklabels()[0].set_fontweight("bold")
ax.get_xticklabels()[5].set_fontweight("bold")

# Title of the subplot
ax.set_title("Correct/Incorrect submissions", fontweight='bold')
# ================== Subplot 2 ================== #
i = 1
plot2 = plt.subplot2grid((1, 5), (0, 2), colspan=3)

ax = filtered[["team", "precision", "recall"]].plot(x='team', kind='bar', width=0.8, color=["#218380", "#FFBC42"], ax=plot2)

ax.bar_label(ax.containers[0], labels=filtered["precision"].round(2), label_type='edge', fontsize=10)
ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["recall"].round(2)], label_type='edge',fontsize=10)

ax.set_xlabel("Teams")
ax.set_xticklabels(filtered["team"])
ax.yaxis.tick_right()
ax.set_ylabel("Score", fontweight='bold')

ax.get_xticklabels()[0].set_fontweight("bold")
ax.get_xticklabels()[5].set_fontweight("bold")
ax.set_title("Precision/Recall", fontweight='bold')

# Create a single legend for all subplots
# handles, labels = ax.get_legend_handles_labels()
# fig.legend(handles[::-1], ["Incorrect", "Correct"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.title("Performace of all teams in term of correct/incorrect submissions in LSC'23", fontweight='bold')
plt.tight_layout()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("qa_23.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# set figure size
plt.rcParams["figure.figsize"] = (8, 8)

# Plot data for each TASK
filtered = df[df["task"] == "KIS"]
filtered[[f"time_correct_{i}" for i in time_till_correct["MyEachtra"] if task_type(i) == "KIS"]].T.plot(kind='box',
        showmeans=False, showfliers=True,
        boxprops=dict(facecolor="#8da0cb", color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker='o', markersize=6, color="#8da0cb"),
        patch_artist=True)

plt.xlabel("Team", fontweight='bold')
plt.xticks(ticks=range(1, 15), labels=filtered["team"], rotation=90)
# set xtick label 0 and 6 in bold
plt.gca().get_xticklabels()[0].set_fontweight("bold")
plt.gca().get_xticklabels()[5].set_fontweight("bold")

# ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')f
# make y-axis ends at 12
# ax.set_ylim(0, 12)
plt.axhline(y=limits['KIS'], color='r', linestyle='-', label=f"Time limit: {limits['KIS']}s")
# Turn off legend
fig.legend().set_visible(False)

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
plt.title("Time to correct submission for KIS task in LSC'23.", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("time_23.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# set figure size
plt.rcParams["figure.figsize"] = (8, 8)

# Plot data for each TASK
filtered = df[df["task"] == "QA"]
filtered[[f"time_correct_{i}" for i in time_till_correct["MyEachtra"] if task_type(i) == "QA"]].T.plot(kind='box',
        showmeans=False, showfliers=True,
        boxprops=dict(facecolor="#8da0cb", color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker='o', markersize=6, color="#8da0cb"),
        patch_artist=True)

plt.xlabel("Team", fontweight='bold')
plt.xticks(ticks=range(1, 15), labels=filtered["team"], rotation=90)
# set xtick label 0 and 6 in bold
plt.gca().get_xticklabels()[0].set_fontweight("bold")
plt.gca().get_xticklabels()[5].set_fontweight("bold")

# ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')f
# make y-axis ends at 12
# ax.set_ylim(0, 12)
plt.axhline(y=limits['QA'], color='r', linestyle='-', label=f"Time limit: {limits['QA']}s")
# Turn off legend
fig.legend().set_visible(False)

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
plt.title("Time to correct submission for QA task in LSC'23.", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("time_qa_23.png", format="png", bbox_inches='tight')

In [ ]:
filtered[[f"time_correct_{i}" for i in time_till_correct["MyEachtra"] if task_type(i) == "QA"]].T.describe()


In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# Create subplots
fig, axes = plt.subplots(1, len(task_names), figsize=(5 + len(task_names) * 5, 8))

# Plot data for each TASK
for i, TASK in enumerate(task_names):
    filtered = df[df["task"] == TASK]
    ax = filtered[[f"time_correct_{i}" for i in time_till_correct["MyEachtra"] if task_type(i) == TASK]].T.plot(kind='box',
        showmeans=False, showfliers=True,
        boxprops=dict(facecolor="#8da0cb", color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker='o', markersize=6, color="#8da0cb"),
        patch_artist=True,
        ax=axes[i])
    
    # ax.bar_label(ax.containers[0], labels=filtered["precision"], label_type='center')
    # ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["recall"]], label_type='center')
    
    ax.set_xlabel("Teams")
    # ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
    ax.set_xticklabels(filtered["team"])
    if i == 0:
        ax.text(-0.12, 0.5, "Seconds", va='center', rotation='vertical', transform=ax.transAxes)

    # set rotation of x-axis labels
    ax.tick_params(axis='x', rotation=90)
    
    # add a horizontal line for the limit (with label)
    ax.axhline(y=limits[TASK], color='r', linestyle='-', label=f"Time limit: {limits[TASK]}s")

    if TASK == "KIS":
        ax.set_ylim(0, 310)
    elif TASK == "QA":
        ax.set_ylim(0, 200)
    else:
        ax.set_ylim(0, 200)
    
    # Turn off legend
    ax.legend().set_visible(False)

    # Title of the subplot
    ax.set_title(f"{TASK}", fontweight='bold')
    


# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
fig.suptitle("Time to find a correct submission per team in LSC'23", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("time_23.png", format="png", bbox_inches='tight')

In [ ]:
# !pip install -U matplotlib
import matplotlib.pyplot as plt
sns.set_style("white")
plt.rcParams.update({'font.size': 16})

# Create subplots
fig, axes = plt.subplots(1, len(task_names), figsize=(5 + len(task_names) * 5, 8))

# Plot data for each TASK
for i, TASK in enumerate(task_names):
    filtered = df[df["task"] == TASK]
    ax = filtered[[f"time_correct_full_{i}" for i in time_till_correct["MyEachtra"] if task_type(i) == TASK]].T.plot(kind='box',
        showmeans=False, showfliers=True,
        boxprops=dict(facecolor="#8da0cb", color="black", linewidth=1),
        medianprops=dict(color="black", linewidth=1),
        whiskerprops=dict(color="black", linewidth=1),
        capprops=dict(color="black", linewidth=1),
        flierprops=dict(marker='o', markersize=6, color="#8da0cb"),
        patch_artist=True,
        ax=axes[i])
    
    # ax.bar_label(ax.containers[0], labels=filtered["precision"], label_type='center')
    # ax.bar_label(ax.containers[1], labels=["" if x == 0 else x for x in filtered["recall"]], label_type='center')
    # add a horizontal line for the limit (with label)
    ax.axhline(y=limits[TASK], color='r', linestyle='-', label=f"Time limit: {limits[TASK]}s")

    if TASK == "KIS":
        ax.set_ylim(0, 310)
    elif TASK == "QA":
        ax.set_ylim(0, 200)
    else:
        ax.set_ylim(0, 200)
    
    ax.set_xlabel("Teams")
    # ax.set_ylabel("Number of {} queries".format(TASK), fontweight='bold')
    ax.set_xticklabels(filtered["team"])
    if i == 0:
        ax.text(-0.12, 0.5, "Seconds", va='center', rotation='vertical', transform=ax.transAxes)

    # set rotation of x-axis labels
    ax.tick_params(axis='x', rotation=90)
    
    # Turn off legend
    ax.legend().set_visible(False)

    # Title of the subplot
    ax.set_title(f"{TASK}", fontweight='bold')

# Create a single legend for all subplots
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, ["Precision", "Recall"], loc='upper right', bbox_to_anchor=(1, 1), frameon=False, prop={'weight': 'bold'})
plt.subplots_adjust(top=0.9)  # Adjust the top margin for the suptitle
fig.suptitle("Time to find a correct submission per team in LSC'23", fontweight='bold')

# Add spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

print()
# plt.savefig("incorrect_correct.eps", format="eps", bbox_inches='tight', dpi=1200)
plt.savefig("time_full_23.png", format="png", bbox_inches='tight')